In [5]:
!pip install requests tqdm openpyxl

In [2]:
import pandas as pd

dfpr = pd.read_pickle("dfpr_with_embeddings.pkl")

In [6]:
import time
import requests
import pandas as pd
from tqdm.auto import tqdm

c:\Users\user\Desktop\Курсач\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
VM_URL = "http://158.160.166.67:8000"

RANKING_URL = f"{VM_URL}/ranking"
HEALTHY_URL = f"{VM_URL}/healthy"

OUT_PATH = "vm_ranking_results_top10.xlsx"

In [8]:
r = requests.get(HEALTHY_URL, timeout=30)
print(r.status_code)
print(r.json())

200
{'status': 'ok', 'catboost_loaded': True, 'embedder_loaded': True, 'accounts_loaded': True, 'accounts_count': 284, 'model_features_count': 90, 'model_name': 'BAAI/bge-m3', 'normalize_embeddings': True}


In [10]:
def build_payload(row):
    return {
        "product": str(row["Продукт / Услуга"]) if pd.notna(row["Продукт / Услуга"]) else "",
        "video_format": str(row["Формат видео"]) if pd.notna(row["Формат видео"]) else "",
        "tone_of_voice": str(row["Tone of Voice (ToV)"]) if pd.notna(row["Tone of Voice (ToV)"]) else "",
        "brief": str(row["Описание рекламной кампании (Бриф для ИИ)"]) if pd.notna(row["Описание рекламной кампании (Бриф для ИИ)"]) else "",
    }


def request_ranking(session, payload, timeout=180, retries=3, sleep_sec=3):
    last_error = None
    last_status_code = None

    for attempt in range(1, retries + 1):
        try:
            response = session.post(
                RANKING_URL,
                json=payload,
                timeout=timeout,
            )

            last_status_code = response.status_code

            if response.status_code == 200:
                return response.json(), response.status_code

            last_error = f"HTTP {response.status_code}: {response.text[:1000]}"

        except Exception as e:
            last_error = f"{type(e).__name__}: {str(e)}"

        if attempt < retries:
            time.sleep(sleep_sec)

    raise RuntimeError(last_error)

In [11]:
all_rows = []
errors = []
request_logs = []

session = requests.Session()

total_start = time.perf_counter()

for campaign_original_index, row in tqdm(dfpr.iterrows(), total=len(dfpr)):
    payload = build_payload(row)

    request_start = time.perf_counter()

    status = "ok"
    http_status = None
    rows_returned = 0
    error_text = None

    try:
        result, http_status = request_ranking(session, payload)

        elapsed_sec = time.perf_counter() - request_start

        top = result.get("top", [])
        rows_returned = len(top)

        for rank, blogger in enumerate(top, start=1):
            all_rows.append({
                "campaign_original_index": campaign_original_index,
                "campaign_rank": rank,

                "campaign_product": payload["product"],
                "campaign_video_format": payload["video_format"],
                "campaign_tone_of_voice": payload["tone_of_voice"],
                "campaign_brief": payload["brief"],
                "rendered_campaign_text": result.get("rendered_campaign_text"),

                "Nickname": blogger.get("Nickname"),
                "Link": blogger.get("Link"),
                "Вовлеченность ERR": blogger.get("user_ERR"),
                "Средние просмотры": blogger.get("avg_views"),
                "Средние лайки": blogger.get("avg_likes"),
                "Виральность": blogger.get("viral_koef"),
                "AI-Score": blogger.get("AI_Score"),
                "Vibe": blogger.get("Vibe"),
                "ToV": blogger.get("ToV"),
                "Бренды": blogger.get("brands"),
            })

    except Exception as e:
        elapsed_sec = time.perf_counter() - request_start

        status = "error"
        error_text = str(e)

        errors.append({
            "campaign_original_index": campaign_original_index,
            "campaign_product": payload["product"],
            "error": error_text,
        })

        print(f"ERROR campaign={campaign_original_index}: {e}")

    request_logs.append({
        "campaign_original_index": campaign_original_index,
        "campaign_product": payload["product"],
        "status": status,
        "http_status": http_status,
        "rows_returned": rows_returned,
        "elapsed_sec": elapsed_sec,
        "error": error_text,
    })

total_elapsed_sec = time.perf_counter() - total_start

results_df = pd.DataFrame(all_rows)
errors_df = pd.DataFrame(errors)
request_logs_df = pd.DataFrame(request_logs)

100%|██████████| 43/43 [00:52<00:00,  1.22s/it]


In [12]:
import numpy as np

In [13]:
ok_logs = request_logs_df[request_logs_df["status"] == "ok"].copy()

perf_summary = pd.DataFrame([{
    "total_requests": len(request_logs_df),
    "ok_requests": len(ok_logs),
    "error_requests": len(request_logs_df) - len(ok_logs),

    "total_elapsed_sec": total_elapsed_sec,
    "overall_rps": len(request_logs_df) / total_elapsed_sec if total_elapsed_sec > 0 else np.nan,
    "ok_rps": len(ok_logs) / total_elapsed_sec if total_elapsed_sec > 0 else np.nan,

    "latency_mean_sec": ok_logs["elapsed_sec"].mean(),
    "latency_median_sec": ok_logs["elapsed_sec"].median(),
    "latency_p90_sec": ok_logs["elapsed_sec"].quantile(0.90),
    "latency_p95_sec": ok_logs["elapsed_sec"].quantile(0.95),
    "latency_min_sec": ok_logs["elapsed_sec"].min(),
    "latency_max_sec": ok_logs["elapsed_sec"].max(),
}])

perf_summary

,total_requests,ok_requests,error_requests,total_elapsed_sec,overall_rps,ok_rps,latency_mean_sec,latency_median_sec,latency_p90_sec,latency_p95_sec,latency_min_sec,latency_max_sec
0,43,43,0,52.574478,0.817887,0.817887,1.220965,1.18417,1.245201,1.252341,1.062204,3.128024


In [14]:
with pd.ExcelWriter(OUT_PATH, engine="openpyxl") as writer:
    results_df.to_excel(writer, sheet_name="rankings_top10", index=False)

    check_counts = (
        results_df
        .groupby("campaign_original_index")
        .size()
        .rename("n_bloggers")
        .reset_index()
    )
    check_counts.to_excel(writer, sheet_name="campaign_counts", index=False)

    request_logs_df.to_excel(writer, sheet_name="request_logs", index=False)
    perf_summary.to_excel(writer, sheet_name="performance", index=False)

    if len(errors_df) > 0:
        errors_df.to_excel(writer, sheet_name="errors", index=False)

print("Saved:", OUT_PATH)

Saved: vm_ranking_results_top10.xlsx


In [20]:
results_df.groupby("Link").count().sort_values("campaign_original_index", ascending=False)

,campaign_original_index,campaign_rank,campaign_product,campaign_video_format,campaign_tone_of_voice,campaign_brief,rendered_campaign_text,Nickname,Вовлеченность ERR,Средние просмотры,Средние лайки,Виральность,AI-Score,Vibe,ToV,Бренды
Link,,,,,,,,,,,,,,,,
https://www.instagram.com/liliitom?igsh=YXM4cHR4bXM5eXFk,37,37,37,37,37,37,37,37,37,37,37,37,37,37,37,37
https://www.instagram.com/yurieva_yuliya?igsh=MXFnYjIybmEyMXh2NQ==,35,35,35,35,35,35,35,35,35,35,35,35,35,35,35,35
https://www.instagram.com/msv_01_01_01?igsh=ZTFkYmgybDd4bTV3,32,32,32,32,32,32,32,32,32,32,32,32,32,32,32,32
https://www.instagram.com/mokka.ira/,27,27,27,27,27,27,27,27,27,27,27,27,27,27,27,27
https://www.instagram.com/kriv.da?igsh=MXd2ZjBoZmtudGZsaw==,27,27,27,27,27,27,27,27,27,27,27,27,27,27,27,27
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
https://www.instagram.com/pavlowna?igsh=dzJ2ZjI2cnFqOGY2,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
https://www.instagram.com/ryzh.nika/,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
https://www.instagram.com/tochaaaaaaaaaa?igsh=MWE1M3p5eDBuajMxcg%3D%3D&utm_source=qr,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
